# Episode 01: Vectors, Matrices & The Rank of Information

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning, PEFT & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\textbf{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Master in this Notebook
Fine-tuning literature is full of phrases like *"the update matrix has a low intrinsic rank"* or *"LoRA factorizes weight matrices"*. But what does that actually mean under the hood?

In this notebook, we bridge the conceptual gap between traditional ML and LLMs before descending to the mathematical ground floor:
1. **The Big Picture:** What fine-tuning is, how it evolved from classic CNNs/BERT to LLMs, why it is difficult, and why it is indispensable.
2. **Tokens as Vectors:** What a word embedding actually looks like in numbers.
3. **Linear Transformations:** What actually happens when a token meets a weight matrix ($y = x \cdot W^T$).
4. **The Intuition of Matrix Rank:** What "rank" actually measures (informational independence vs redundancy) using small $2 \times 2$ and $3 \times 3$ matrices.
5. **The Low-Rank Factorization Trick:** How a $3 \times 3$ matrix of 9 numbers can be represented by 6 numbers ($B \times A$).
6. **Scaling to LLM Dimensions:** How this simple matrix factorization saves millions of parameters in models like `Qwen` and `Llama`.


---

## 0. The Big Picture: What is Fine-Tuning & Why Does it Matter?

Before diving into the linear algebra of weight matrices and low-rank factorization, let's step back and establish the big picture: **What is fine-tuning, how has it evolved, why is it so notoriously difficult for large models, and why is it an indispensable superpower for AI engineers?**

---

### 1. What is Fine-Tuning? (The Two-Phase Paradigm)

Modern machine learning is built on a two-phase philosophy called **Transfer Learning**:

$$\text{Phase 1: Pre-Training (Generalist)} \longrightarrow \text{Phase 2: Fine-Tuning (Specialist)}$$

* **Phase 1: Pre-Training:**  
  A model reads trillions of tokens across the open web (books, articles, code, conversations). It spends millions of dollars of compute learning the statistical structure of language, world facts, coding syntax, and common-sense reasoning. The result is a **foundation model**—knowledgeable, but broad, unguided, and prone to rambling.
* **Phase 2: Fine-Tuning:**  
  You take that pre-trained model and continue training it on a much smaller, curated dataset ($100$ to $10,000$ high-quality examples) tailored to a specific task, format, or persona. You are **not** teaching the model English from scratch; you are steering its pre-existing capabilities toward your exact requirements.

---

### 2. How Fine-Tuning Was Done in Classic Neural Networks

In the pre-LLM era (Computer Vision and early NLP models like ResNet, VGG, or BERT), fine-tuning was straightforward because models were relatively small (typically $10\text{M}$ to $300\text{M}$ parameters):

```text
Classic Neural Network (e.g., ResNet-50 / BERT-Base: ~100M parameters)
┌──────────────────────────────────────────────┐
│ Layer 1 .. Layer N-1 (Feature Extractor)     │  <-- FROZEN (weights locked)
│ [Convolutions / Self-Attention Layers]       │
└──────────────────────────────────────────────┘
                      │
                      ▼
┌──────────────────────────────────────────────┐
│ Layer N: Final Classification Head           │  <-- REPLACED & TRAINED
│ [Linear(768, 2) -> "Cat" vs "Dog"]           │      (Takes 5 minutes on a laptop)
└──────────────────────────────────────────────┘
```

**The Classic Recipe:**
1. **Freeze the backbone:** Lock the weights of the earlier layers (which act as universal feature extractors for edges, shapes, or basic grammar).
2. **Chop off the final head:** Discard the original 1000-class output layer.
3. **Attach a new head:** Plug in a tiny linear layer matching your new task (e.g. 2 classes for sentiment classification).
4. **Train only the head:** Backpropagate gradients strictly into those few thousand parameters.

This was fast, cheap, and fit comfortably on a single consumer GPU or CPU in minutes.

---

### 3. Then Came LLMs: Why the Classic Approach Broke Down

When Large Language Models (LLMs) like GPT-3, Llama, and Qwen arrived, the classic "chop off the head" recipe completely fell apart for two fundamental reasons:

1. **Scale Explosion ($100\text{M} \longrightarrow 8\text{B} - 70\text{B}+$):**  
   Models grew by a factor of $100\times$ to $700\times$. You can no longer just hold the model in GPU memory and calculate gradients casually.
2. **Generative Autoregressive Nature (No Single "Head"):**  
   An LLM is not a simple classifier that outputs a single label. It is an **autoregressive sequence generator**. Every single token it outputs must pass through all 32+ layers of attention and feed-forward networks. The model's "head" is an enormous vocabulary projection matrix ($W_{\text{vocab}} \approx 150,000$ tokens), and the reasoning, style, formatting, and grammar are distributed **throughout every layer of the network**. You cannot simply "replace the last layer" to change its behavior.

---

### 4. Why is LLM Fine-Tuning Difficult? (The 3 Hard Engineering Hurdles)

If you attempt to fine-tune an LLM by updating all of its weights (known as **Full Fine-Tuning / FFT**), you immediately collide with three massive engineering bottlenecks:

#### 🛑 Hurdle 1: The Memory Wall (The 16-to-20 Bytes/Param Rule)
When you train a neural network, you don't just store the model's weights. You must also store:
- **Gradients:** The rate of change for every single weight tensor ($\nabla_W \mathcal{L}$).
- **Optimizer States:** Modern optimizers like AdamW maintain two running tracking buffers (momentum $m$ and variance $v$) for *every single parameter*.
- **Activations:** The intermediate outputs of every layer needed during backpropagation.

$$\textbf{Static Training Footprint} \approx \mathbf{16 \text{ to } 20 \text{ Bytes per Parameter}}$$

For an 8-Billion parameter model (which is only $\approx 16\text{ GB}$ on disk):
$$8\text{B} \times 16\text{ bytes} \approx \mathbf{128\text{ GB of VRAM}}$$
You cannot train it on a consumer GPU ($16\text{–}24\text{ GB}$). You need a multi-GPU cloud cluster of enterprise A100/H100 GPUs ($80\text{ GB}$ each) just to hold the model and its optimizer states!

#### 🧠 Hurdle 2: Catastrophic Forgetting
When you forcefully update billions of pre-trained weights on a small dataset (e.g., 500 company support tickets), the gradient updates can easily overwrite the fragile, intricate connections that give the model its general intelligence. The model might learn your customer service format, but completely lose its ability to write code, do basic math, or follow complex multi-turn logic.

#### ⚡ Hurdle 3: Format & Alignment Brittleness
Unlike standard software code, LLMs are probabilistic token predictors. If you train on poorly formatted prompts, misalign special tokens (like `<|im_start|>` or `<|im_end|>`), or calculate loss on user instructions instead of assistant responses, the model will collapse into echoing user questions (the infamous **Prompt Parrot Bug**) or leaking raw markdown fences.

---

### 5. Why is LLM Fine-Tuning Needed? (When Prompting & RAG Hit a Wall)

Given these hurdles, why do AI engineers still fine-tune models? Why not just use Prompt Engineering or RAG (Retrieval-Augmented Generation)?

Because **Prompting, RAG, and Fine-Tuning solve fundamentally different architectural problems**:

| Capability | Prompt Engineering | RAG (Retrieval) | Fine-Tuning (SLM / LoRA) |
|---|:---:|:---:|:---:|
| **Fresh / Dynamic Facts** (e.g., "What is today's inventory?") | ❌ No | ✅ **Best** (Fetch latest data) | ❌ Poor (Knowledge is static in weights) |
| **Strict Grammar / JSON / DSL Reliability** ($99.9\%+$ contract) | ⚠️ Unreliable (hallucinates tags) | ⚠️ Unreliable | ✅ **Best** (Burned into neural reflexes) |
| **Latency & Throughput** (Sub-$50\text{ms}$ response times) | ❌ Slow (massive prompt tokens) | ❌ Slow (vector search + context) | ✅ **Best** (Short prompt, tiny model) |
| **Cost at High Scale** ($1\text{M}+$ requests/month) | ❌ Expensive (frontier API fees) | ❌ High API & vector DB costs | ✅ **Best** (In-house $0.5\text{B}$ model runs for pennies) |
| **Custom Persona, Brand Tone & Brevity** | ⚠️ Needs long prompt examples | ⚠️ Weak | ✅ **Best** (Permanent behavioral reflex) |

> **The Production Golden Rule:**  
> - Use **RAG** when the model lacks **facts** or the data changes daily.  
> - Use **Prompting** when you are prototyping and discovering user requirements.  
> - Use **Fine-Tuning** when you need **consistent form, strict JSON, specialized behavior, low latency, and low operational cost**.

---

### 6. The Breakthrough: The Low-Rank Adaptation (LoRA) Insight

To eliminate the **Memory Wall** and prevent **Catastrophic Forgetting**, researchers asked a profound mathematical question:
> *"When adapting an 8-Billion parameter model to a specific task, do we really need to change all 8 Billion degrees of freedom?"*

The answer is **NO**. The necessary change in the weight matrix ($\Delta W$) has an **extremely low intrinsic rank**.

By decomposing massive weight updates into small, low-rank matrix pairs, **LoRA (Low-Rank Adaptation)** reduces trainable parameters by **$99.6\%+$**, allowing you to fine-tune modern models on a single GPU or laptop without touching the pre-trained weights.

To understand how and why that works mathematically, we must start from the very foundation: **What is a weight matrix, what is a token vector, and what is Matrix Rank?** Let's dive in.


## 1. Setup & Deterministic Environment

We only need standard `numpy` and `torch`. If on a Mac, PyTorch will use CPU or Metal (MPS); every calculation in this notebook executes instantaneously.


In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)
print("Environment ready. PyTorch version:", torch.__version__)


---

## 2. Absolute Basics: A Token is Just a Vector

In a large language model, text is broken into tokens, and each token is converted into a vector of numbers called an **embedding**.

For example, suppose the word `"server"` is embedded into a tiny 3-dimensional vector:
$$x = \begin{bmatrix} 1.0 & 2.0 & 3.0 \end{bmatrix}$$

Each number represents a coordinate in semantic feature space.


In [ ]:
# 1 token, represented by 3 features
x = torch.tensor([[1.0, 2.0, 3.0]])
print("Token vector x shape:", x.shape)
print("Token vector x values:", x.numpy())


---

## 3. What Happens When a Token Meets a Weight Matrix?

Inside every attention head and MLP block of an LLM, the input token vector $x$ is transformed by multiplying against a pre-trained weight matrix $W_0$:

$$y = x \cdot W_0^T$$

Let's bring in our token vector $x$ from earlier and inspect every dimension carefully:

$$x = \begin{bmatrix} 1.0 & 2.0 & 3.0 \end{bmatrix} \quad \implies \text{Shape: } (1, 3) \quad (1 \text{ token, } d_{\text{in}} = 3 \text{ features})$$

Now, let's define a $3 \times 3$ pre-trained weight matrix $W_0$:

$$W_0 = \begin{bmatrix} 1 & 0 & 2 \\ 0 & 3 & 1 \\ 2 & 1 & 0 \end{bmatrix} \quad \implies \text{Shape: } (3, 3) \quad (d_{\text{out}} = 3, d_{\text{in}} = 3)$$

---

### 📐 The Golden Rule of Matrix Multiplication (Inner Dimensions Must Match)

In linear algebra, you can only multiply two matrices if their **inner dimensions match**:

$$\underbrace{(m \times k)}_{A} \times \underbrace{(k \times n)}_{B} = \underbrace{(m \times n)}_{\text{Result}}$$

If you try to multiply $(1 \times 3)$ by $(3 \times 3)$, the inner dimensions ($3$ and $3$) match, resulting in a $(1 \times 3)$ vector.

---

### 🤔 Why the Transpose ($W_0^T$)? Why Not Just $x \cdot W_0$?

This is one of the most common questions from engineers learning deep learning:

1. **How PyTorch Stores Weights:**  
   In PyTorch, a linear layer (`nn.Linear(in_features, out_features)`) stores its weight matrix in the shape:
   $$\text{Shape}(W) = (d_{\text{out}}, d_{\text{in}})$$
   Why? Because **Row $i$ of $W$ contains the weights that compute Output Feature $i$**.

2. **What Happens When $d_{\text{in}} \ne d_{\text{out}}$?**  
   Suppose our input token has $3$ features ($d_{\text{in}} = 3$), but we want to project it to $5$ output features ($d_{\text{out}} = 5$):
   - Token $x$ has shape: $(1, 3)$
   - Weight $W$ has shape: $(5, 3)$
   
   If you attempted $x \cdot W$:
   $$(1 \times 3) \cdot (5 \times 3) \implies \textbf{CRASH! (Inner dimensions 3 and 5 do not match!)}$$

   To make the dimensions align, we **transpose $W$** ($W^T$), flipping its shape from $(5, 3)$ to $(3, 5)$:
   $$\underbrace{(1 \times 3)}_{x} \cdot \underbrace{(3 \times 5)}_{W^T} = \underbrace{(1 \times 5)}_{y} \quad \checkmark \text{ Inner dimensions match!}$$

This is why PyTorch's `F.linear(x, weight)` is officially defined as:
$$\text{output} = x \cdot W^T + b$$

---

### 🔢 Step-by-Step Arithmetic

In our $3 \times 3$ example, $W_0^T$ is:

$$W_0^T = \begin{bmatrix} 1 & 0 & 2 \\ 0 & 3 & 1 \\ 2 & 1 & 0 \end{bmatrix}$$

Multiplying row vector $x = [1.0, 2.0, 3.0]$ against each column of $W_0^T$ (which is row $i$ of $W_0$):
- **Output 1:** $1(1) + 2(0) + 3(2) = 1 + 0 + 6 = \mathbf{7.0}$
- **Output 2:** $1(0) + 2(3) + 3(1) = 0 + 6 + 3 = \mathbf{9.0}$
- **Output 3:** $1(2) + 2(1) + 3(0) = 2 + 2 + 0 = \mathbf{4.0}$

Result:
$$y = \begin{bmatrix} 7.0 & 9.0 & 4.0 \end{bmatrix}$$


In [ ]:
import torch.nn.functional as F

# 1. Explicitly define token vector x: Shape (batch_size=1, d_in=3)
x = torch.tensor([[1.0, 2.0, 3.0]])

# 2. Define pre-trained weight matrix W_0: Shape (d_out=3, d_in=3)
W_0 = torch.tensor([
    [1.0, 0.0, 2.0],  # Weights for output feature 1
    [0.0, 3.0, 1.0],  # Weights for output feature 2
    [2.0, 1.0, 0.0]   # Weights for output feature 3
])

print(f"Token x shape:          {x.shape} (1 token with d_in=3)")
print(f"Weight W_0 shape:       {W_0.shape} (d_out=3, d_in=3)")
print(f"Transposed W_0.T shape: {W_0.T.shape} (d_in=3, d_out=3)")

# 3. Compute linear transformation: (1, 3) @ (3, 3) -> (1, 3)
y = x @ W_0.T
print("\nProjected Output y (x @ W_0.T):", y.numpy())

# 4. Compare with PyTorch's native F.linear (proves it computes x @ W.T under the hood!)
y_native = F.linear(x, W_0)
print("PyTorch F.linear(x, W_0):        ", y_native.numpy())
print("Exact match?", torch.equal(y, y_native))


---

## 4. What on Earth is Matrix "Rank"?

Now comes the fundamental concept behind all parameter-efficient fine-tuning (PEFT): **Rank**.

> **Definition:** The **Rank** of a matrix is the maximum number of **linearly independent** rows or columns it contains.  
> **In Plain English:** Rank is the **effective information dimension** of the matrix. It tells you how many genuinely unique patterns are stored inside.

Let's see this with two concrete $3 \times 3$ examples.

### Example A: Full Rank (Rank = 3)
Look at this identity matrix:
$$M_{\text{full}} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

Row 1 points strictly along the X-axis (`[1, 0, 0]`).  
Row 2 points strictly along the Y-axis (`[0, 1, 0]`).  
Row 3 points strictly along the Z-axis (`[0, 0, 1]`).  

None of these rows can be created by scaling or adding the other rows. All 3 directions are completely independent.  
Therefore, **Rank = 3 (Full Rank)**.

---

### Example B: Redundant / Low Rank (Rank = 1)
Now look at this matrix:

$$M_{\text{low}} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

Notice the pattern:
- **Row 2** is just **$2 \times \text{Row 1}$** ($[1, 2, 3] \times 2 = [2, 4, 6]$).
- **Row 3** is just **$3 \times \text{Row 1}$** ($[1, 2, 3] \times 3 = [3, 6, 9]$).

Even though there are **9 separate numbers** written in this matrix, there is really only **ONE unique direction of information** ($[1, 2, 3]$). The other two rows are completely redundant!  
Therefore, **Rank = 1**.


In [ ]:
M_full = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

M_low = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [3.0, 6.0, 9.0]
])

rank_full = torch.linalg.matrix_rank(M_full).item()
rank_low  = torch.linalg.matrix_rank(M_low).item()

print(f"M_full rank: {rank_full} -> 3 independent directions of information.")
print(f"M_low rank:  {rank_low} -> Only 1 independent direction! (8 of the numbers are redundant copies)")


---

## 5. The Low-Rank Factorization Magic: $9$ numbers into $6$

Because $M_{\text{low}}$ has **Rank 1**, we do not need to store all 9 numbers!

Any rank-1 matrix of size $3 \times 3$ can be **decomposed (factorized)** into the outer product of two skinny vectors:
- A column vector $B$ of size $3 \times 1$
- A row vector $A$ of size $1 \times 3$

Let's test this arithmetic:

$$B = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}, \quad A = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$$

Multiply $B \times A$:

$$B \times A = \begin{bmatrix} 1 \cdot 1 & 1 \cdot 2 & 1 \cdot 3 \\ 2 \cdot 1 & 2 \cdot 2 & 2 \cdot 3 \\ 3 \cdot 1 & 3 \cdot 2 & 3 \cdot 3 \end{bmatrix} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

Look at the storage comparison:
- Storing the full matrix: $3 \times 3 = 9$ numbers.
- Storing $B$ and $A$: $3 + 3 = \mathbf{6}$ numbers.
- **You saved $33\%$ of storage while reproducing the matrix perfectly.**


In [ ]:
B = torch.tensor([[1.0], [2.0], [3.0]])  # Shape: (3, 1)
A = torch.tensor([[1.0, 2.0, 3.0]])       # Shape: (1, 3)

# Matrix multiplication: (3, 1) @ (1, 3) -> (3, 3)
reconstructed = B @ A

print("B (3x1):\n", B.numpy())
print("\nA (1x3):\n", A.numpy())
print("\nProduct B @ A (3x3):\n", reconstructed.numpy())
print("\nExact match with M_low?", torch.equal(reconstructed, M_low))


---

## 6. Scaling to Real LLMs: Why Rank Saves Billion-Parameter Models

Now scale this exact insight up to modern transformer architectures like `Qwen2.5-7B` or `Llama-3-8B`:
- In these models, a single attention weight matrix has dimension:
  $$d_{\text{in}} = 4096, \quad d_{\text{out}} = 4096$$

### Full Matrix:
$$4096 \times 4096 = \mathbf{16,777,216 \text{ parameters}}$$
In standard 16-bit float, a single weight matrix takes **$33.5\text{ MB}$**.

### LoRA Factorization with Rank $r = 8$:
Instead of training a full $4096 \times 4096$ update matrix $\Delta W$, we decompose it into:
- $A \in \mathbb{R}^{8 \times 4096} = 32,768$ parameters
- $B \in \mathbb{R}^{4096 \times 8} = 32,768$ parameters
- **Total Parameters:** $32,768 + 32,768 = \mathbf{65,536 \text{ parameters}}$

$$\frac{65,536}{16,777,216} = 0.0039 \implies \mathbf{99.61\% \text{ reduction in parameters!}}$$


In [ ]:
def print_rank_scaling_table(d_in=4096, d_out=4096):
    full_params = d_in * d_out
    full_mb = (full_params * 2) / (1024**2) # FP16
    
    print(f"Base Matrix Dimension: {d_in} x {d_out}")
    print(f"Full Update Params:    {full_params:,} ({full_mb:.2f} MB)\n")
    print(f"{'Rank (r)':<10} | {'LoRA Params':<15} | {'Memory (FP16)':<15} | {'Savings %':<10}")
    print("-" * 60)
    
    for r in [1, 2, 4, 8, 16, 32, 64]:
        lora_params = (d_in * r) + (r * d_out)
        lora_mb = (lora_params * 2) / (1024**2)
        savings = (1 - lora_params / full_params) * 100
        print(f"{r:<10} | {lora_params:<15,} | {lora_mb:<12.3f} MB | {savings:.2f}%")

print_rank_scaling_table(4096, 4096)


---

## 🎯 Episode 01 Comfort Recap

1. **A token is a vector** representing coordinates in semantic feature space.
2. **A linear layer** computes $y = x W^T$, transforming token coordinates into new representations.
3. **Matrix Rank** measures the number of genuinely independent directions of information.
4. **Low-Rank Factorization ($B \times A$):** If an update $\Delta W$ has a low intrinsic rank, we can decompose a large $d \times k$ matrix into two skinny matrices $B (d \times r)$ and $A (r \times k)$.
5. **Why this matters for LLMs:** Adapting an LLM for formatting, tone, or tool calling only requires rank $r=8$ or $r=16$, slashing trainable parameters by **$99.6\%$**.

---

### Up Next:
In **Episode 02: Full Fine-Tuning (FFT) vs PEFT & Memory Math**, we explore what happens when we *don't* use low rank:
- The 2D Orthogonal Grid (Objective vs Weight Update).
- Why FFT requires $16\text{ bytes}$ per parameter (exploding an 8B model into $96+\text{ GB}$ VRAM).
- The exact hardware math formula you need before launching any training run.
